In [1]:
import pandas as pd
import os
import rasterio 
from tqdm import tqdm
import numpy as np

os.chdir('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity')

In [2]:
# Step 1: Load the Parquet file
input_file = "LME_boundaries_2014/merged_all_lme.parquet"
df = pd.read_parquet(input_file)
# Step 2: Extract unique station locations
unique_stations = df[['decimalLatitudeStart', 'decimalLongitudeStart']].drop_duplicates()
# Step 3: Save the result to a new file
output_file = "GEBCO/unique_coordinates.parquet"
unique_stations.to_parquet(output_file, index=False)
print(f"Unique station locations saved to {output_file}")

Unique station locations saved to GEBCO/unique_coordinates.parquet


In [9]:
def extract_raster_values(raster_path, coordinates_df, batch_size=10000):
    """
    Extract raster values for given coordinates from a single raster file.
    Parameters:
        raster_path (str): Path to the merged raster file.
        coordinates_df (pd.DataFrame): DataFrame with 'decimalLatitudeStart' and 'decimalLongitudeStart'.
        batch_size (int): Number of points to process in each batch.
    Returns:
        pd.DataFrame: DataFrame with extracted raster values.
    """
    results = []
    total_batches = len(coordinates_df) // batch_size + (1 if len(coordinates_df) % batch_size != 0 else 0)
    with rasterio.open(raster_path) as src:
        bounds = src.bounds  # Get raster bounds
        with tqdm(total=total_batches, desc="Processing Batches", unit="batch") as pbar:
            for i in range(0, len(coordinates_df), batch_size):
                batch = coordinates_df.iloc[i:i + batch_size].copy()
                batch.loc[:, 'gebco'] = np.nan  # Initialize the column for raster values
                # Adjust longitudes at the antimeridian
                batch['decimalLongitudeStart'] = batch['decimalLongitudeStart'].apply(
                    lambda lon: -180 if lon == 180 else lon
                )
                # Filter coordinates within the raster bounds
                batch_in_bounds = batch[
                    (batch['decimalLongitudeStart'] >= bounds.left) &
                    (batch['decimalLongitudeStart'] <= bounds.right) &
                    (batch['decimalLatitudeStart'] >= bounds.bottom) &
                    (batch['decimalLatitudeStart'] <= bounds.top)
                ]
                # Extract raster values for these coordinates
                coords = [(lon, lat) for lon, lat in zip(batch_in_bounds['decimalLongitudeStart'], batch_in_bounds['decimalLatitudeStart'])]
                values = list(src.sample(coords))
                # Assign values to the batch DataFrame
                for idx, value in zip(batch_in_bounds.index, values):
                    if value[0] is not None and not np.isnan(value[0]):
                        batch.loc[idx, 'gebco'] = value[0]
                results.append(batch)
                pbar.update(1)
    return pd.concat(results, ignore_index=True)
# File path to the merged raster
raster_file = "GEBCO/gebco_merged.tif"
# Load the unique_coordinates.parquet file
coordinates = pd.read_parquet("GEBCO/unique_coordinates.parquet")
# Ensure decimalLongitude and decimalLatitude are numeric
coordinates['decimalLongitudeStart'] = pd.to_numeric(coordinates['decimalLongitudeStart'], errors='coerce')
coordinates['decimalLatitudeStart'] = pd.to_numeric(coordinates['decimalLatitudeStart'], errors='coerce')
# Drop rows with invalid (NaN) coordinates
coordinates = coordinates.dropna(subset=['decimalLongitudeStart', 'decimalLatitudeStart'])
# Extract raster values
result_df = extract_raster_values(raster_file, coordinates)
# Save the results to a Parquet file
result_df.to_parquet("GEBCO/gebco_depth.parquet", index=False)
print("Results saved as 'GEBCO/gebco_depth.parquet'")

Processing Batches: 100%|██████████| 20/20 [00:18<00:00,  1.06batch/s]

Results saved as 'GEBCO/gebco_depth.parquet'


In [2]:
# Load tables
lme_file = "LME_boundaries_2014/merged_all_lme.parquet"
gebco_depth_file = "GEBCO/gebco_depth.parquet"
out_file = "GEBCO/merged_all_gebco.parquet"

lme_df = pd.read_parquet(lme_file)
gebco_df = pd.read_parquet(gebco_depth_file)

# Ensure join keys are numeric and consistent
lme_df['decimalLatitudeStart'] = pd.to_numeric(lme_df['decimalLatitudeStart'], errors='coerce')
lme_df['decimalLongitudeStart'] = pd.to_numeric(lme_df['decimalLongitudeStart'], errors='coerce')
gebco_df['decimalLatitudeStart'] = pd.to_numeric(gebco_df['decimalLatitudeStart'], errors='coerce')
gebco_df['decimalLongitudeStart'] = pd.to_numeric(gebco_df['decimalLongitudeStart'], errors='coerce')

# Merge on the combined key (left join to preserve original LME rows)
merged = lme_df.merge(
    gebco_df[['decimalLatitudeStart', 'decimalLongitudeStart', 'gebco']],
    on=['decimalLatitudeStart', 'decimalLongitudeStart'],
    how='left'
)

# Update or create gebcoDepth using the joined gebco values
if 'gebcoDepth' in merged.columns:
    merged['gebcoDepth'] = merged['gebcoDepth'].fillna(merged['gebco'])
else:
    merged['gebcoDepth'] = merged['gebco']

# Drop the auxiliary gebco column and save
merged = merged.drop(columns=['gebco'])
merged.to_parquet(out_file, index=False)
print(f"Saved merged file to {out_file}")

Saved merged file to GEBCO/merged_all_gebco.parquet
